In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
"""
Initialize the analysis environment by loading essential modules and setting up
the Python path to access custom analysis functions.
"""

# Enable automatic reloading of modules for interactive development
%load_ext autoreload
%autoreload 2

# Import essential system modules
import sys
from pathlib import Path

# Define the path to custom analysis modules
# Note: Update this path to match your local installation
MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

# Add module path to system path for importing custom functions
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")
print("🔄 Auto-reload enabled for interactive development")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis
🔄 Auto-reload enabled for interactive development


In [2]:
from general_utils import find_ephys_sessions
from ephys_dimension_reduction_tdr import plot_tdr_trace_by_quantile

sessions=find_ephys_sessions()

['ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38',
 'ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35',
 'ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43',
 'ecephys_844036_2026-05-04_16-06-43_sorted_2026-05-09_21-00-45',
 'ecephys_844036_2026-05-05_16-08-08_sorted_2026-05-19_17-42-51',
 'ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13']

In [6]:
import numpy as np
from ephys_dimension_reduction_tdr import tdr_from_psth
from create_psth import load_zarr
from general_utils import smart_read_csv

failed_sessions = []

for session in sessions[2][-6:]:
    try:
        binsize = '0.1'
        time_windows=[[0.3,2],[-1,0]]
  #      latent_vars=['QLearning_L2F1_softmax-choice','QLearning_L2F1_softmax-reward','QLearning_L2F1_softmax-RPE','QLearning_L2F1_softmax-deltaQ','QLearning_L2F1_softmax-sumQ',
  #      'QLearning_L2F1_softmax-chosenQ','QLearning_L2F1_softmax-chosenQ-1','QLearning_L2F1_softmax-unchosenQ-1','QLearning_L2F1_softmax-unchosenQ']
        latent_vars=['QLearning_L1F1_CK1_softmax-sumQ-1','ForagingCompareThreshold-value-1']
        latent_vars=['ForagingCompareThreshold_L1_ResetF_StayBiasT_FixThrT0-value-1']

        for time_window in time_windows:
            for latent_var in latent_vars:
                #time_window=[0.3,2]
                align = "trial_start"
                #latent_var = 'ForagingCompareThreshold-deltaQ-1'
                #latent_var = 'QLearning_L2F1_softmax-deltaQ-1'
                #latent_var = 'QLearning_L2F1_softmax-RPE'
                #latent_var = 'QLearning_L2F1_softmax-choice'
                psth_da = load_zarr(f"/root/capsule/scratch/psth_results/{session}_{binsize}s.zarr")
                df = smart_read_csv(f"/root/capsule/scratch/behavior_summary/behavior_summary-{session}.csv")
                #latent_var = 'QLearning_L2F1_softmax-sumQ-1'
                keep_ids = np.asarray(df['response_trials'][0], dtype=int)
                latent_full = np.asarray(df[latent_var][0], dtype=float)

                out = tdr_from_psth(
                    psth_da,
                    latent=latent_full,
                    align=align,
                    time_window=time_window,
                    include_trials=keep_ids,
                    require_all_ids=True,
                    save_path=f"/root/capsule/scratch/tdr_{session}_{latent_var}_align_{align}_timewindow_{time_window[0]}_{time_window[1]}.zarr",
                    save_format="zarr",
                )

    except Exception as e:
        print(f"❌ Error in session {session}: {e}")
        failed_sessions.append(session)

print("\nAll sessions done.")
if failed_sessions:
    print("Failed sessions:")
    for s in failed_sessions:
        print(f" - {s}")
else:
    print("No errors 🎉")



All sessions done.
No errors 🎉


In [7]:
from behavior_utils import get_fitted_model_names

get_fitted_model_names(session_name='ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13')

No model-fitting results found for subject 844036 on 2026-05-06


In [8]:
# create and save PSTH
from nwb_utils import NWBUtils
from behavior_utils import extract_fitted_data, find_trials,get_fitted_model_names,generate_behavior_summary
from create_psth import extract_neuron_psth_to_zarr
from behavior_utils import generate_behavior_summary_combined

session_name='ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13'
#session_name='764790_2024-12-19_16-11-34'
print(get_fitted_model_names(session_name=session_name))
save_folder="/root/capsule/scratch"
nwb_data,tag=NWBUtils.combine_nwb(session_name=session_name)

No model-fitting results found for subject 844036 on 2026-05-06
None
Found ephys NWB: /root/capsule/data/ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13/nwb/ecephys_844036_2026-05-06_16-27-46_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13/nwb/ecephys_844036_2026-05-06_16-27-46_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/behavior_nwb/844036_2026-05-06_16-27-46.nwb
Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/844036_2026-05-06_16-27-46.nwb
Successfully appended units table to behavior NWB.


In [10]:
# Quick laser-on / laser-off counts
import numpy as np

laser_flags = np.asarray(nwb_data.trials['laser_on_trial'][:], dtype=int)
n_all = len(laser_flags)
n_laser = int((laser_flags == 1).sum())
print(f"Total trials: {n_all}")
print(f"Laser trials (laser_on_trial == 1): {n_laser}")
print(f"Non-laser trials (laser_on_trial == 0): {n_all - n_laser}")


Total trials: 500
Laser trials (laser_on_trial == 1): 65
Non-laser trials (laser_on_trial == 0): 435
